In [13]:
# Core libraries
import numpy as np
import pandas as pd

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Model
from xgboost import XGBClassifier, XGBRegressor

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    r2_score
)

# Model saving
import pickle

# Warning control
import warnings

# Pipeline (very useful for clean ML workflow)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

In [36]:
file_path = "/kaggle/input/datasets/neelakandannc/kaggle-dataset-1/indian_diseases_dataset.csv"

# Load dataset
df = pd.read_csv(file_path)

In [20]:
df.head()

,patient_id,age,age_group,gender,state,city,region,urban_rural,disease_name,disease_category,...,insurance_status,treatment_type,hospital_type,days_hospitalized,treatment_cost_inr,outcome,death_flag,cause_of_death,recovery_days,follow_up_required
0,IND-00000001,70,70-79,Female,Bihar,Patna,East,Semi-Urban,Diarrhea,Waterborne,...,Employer,Medication,Government,21,102.0,Recovering,0,NaN,42.0,Yes
1,IND-00000002,27,20-29,Other,Gujarat,Surat,West,Urban,Chikungunya,Vector-Borne,...,Uninsured,Medication,Government,0,1151.0,Recovered,0,NaN,18.0,Yes
2,IND-00000003,66,60-69,Other,Jharkhand,Bokaro,East,Rural,Hypertension,Non-Communicable,...,Private,Medication,Government,2,89.0,Chronic Management,0,NaN,NaN,Yes
3,IND-00000004,27,20-29,Female,Karnataka,Hubli,South,Rural,Tuberculosis,Infectious,...,Private,Surgery,Government,9,1269.0,Recovering,0,NaN,69.0,Yes
4,IND-00000005,40,40-49,Male,Karnataka,Belagavi,South,Rural,Chronic Kidney Disease,Non-Communicable,...,Ayushman Bharat,Palliative,Private,31,670576.0,Deceased,1,Complications from Chronic Kidney Disease,NaN,No


In [21]:
# Percentage of missing values in each column
missing_percent = df.isnull().mean() * 100

print(missing_percent)

patient_id             0.000
age                    0.000
age_group              0.000
gender                 0.000
state                  0.000
city                   0.000
region                 0.000
urban_rural            0.000
disease_name           0.000
disease_category       0.000
severity               0.000
diagnosis_date         0.000
year                   0.000
month                  0.000
season                 0.000
symptoms               0.000
comorbidity           32.995
smoking_status         0.000
alcohol_use           33.135
bmi                    0.000
blood_group            0.000
income_level           0.000
insurance_status       0.000
treatment_type         0.000
hospital_type          0.000
days_hospitalized      0.000
treatment_cost_inr     0.000
outcome                0.000
death_flag             0.000
cause_of_death        90.890
recovery_days         31.535
follow_up_required     0.000
dtype: float64


In [6]:
df['alcohol_use'].unique()


array([nan, 'Regular', 'Occasional', 'Heavy'], dtype=object)

In [8]:
df['comorbidity'].unique()

array([nan, 'Kidney Disease', 'Heart Disease', 'Multiple', 'Obesity',
       'Hypertension', 'Diabetes'], dtype=object)

In [9]:
df['cause_of_death'].unique()

array([nan, 'Complications from Chronic Kidney Disease',
       'Complications from Stroke', 'Complications from COPD',
       'Complications from Breast Cancer',
       'Complications from Japanese Encephalitis',
       'Complications from Lung Cancer',
       'Complications from Heart Disease',
       'Complications from Leptospirosis', 'Complications from COVID-19',
       'Complications from HIV/AIDS', 'Complications from Kala-Azar',
       'Complications from Pneumonia', 'Complications from Asthma',
       'Complications from Tuberculosis', 'Complications from Filariasis',
       'Complications from Swine Flu (H1N1)',
       'Complications from Typhoid', 'Complications from Hypertension',
       'Complications from Oral Cancer', 'Complications from Hepatitis B',
       'Complications from Thalassemia',
       'Complications from Diabetes (Type 2)',
       'Complications from Sickle Cell Anemia',
       'Complications from Gastroenteritis',
       'Complications from Depression',
 

In [10]:
df['recovery_days'].unique()

array([ 42.,  18.,  nan,  69.,  95.,  98.,  27.,  14.,  24.,  76.,  41.,
         4.,  54.,  16.,  45.,  52.,  15.,  22.,  79.,  20.,  29.,   7.,
        12.,   8.,  88.,  71.,  26.,  70.,   5.,  90.,   9.,  30.,  28.,
        11.,  68.,  17.,  97.,   3.,  23.,  36.,  35.,  10.,  81.,  50.,
        44.,  19.,   6.,  32.,  92.,  13., 100.,  21.,  91.,  72.,  83.,
        25.,  82.,  78.,  94.,  73.,  84.,  43.,  87.,  86.,  46.,  47.,
        60.,  64.,  59.,  77.,  31.,  93.,  34.,  58.,  49.,  89.,  55.,
        37.,  57.,  67.,  74.,  65.,  48.,  63.,  40.,  53.,  33.,  38.,
        61.,  51.,  96.,  39.,  56.,  99.,  75.,  85.,  66.,  62.,  80.])

alcohol_use - map nan to never

comorbidity - map nan to none

cause_of_death - map nan to not dead

recovery_days - map nan to median


In [11]:
df.dtypes

patient_id             object
age                     int64
age_group              object
gender                 object
state                  object
city                   object
region                 object
urban_rural            object
disease_name           object
disease_category       object
severity               object
diagnosis_date         object
year                    int64
month                  object
season                 object
symptoms               object
comorbidity            object
smoking_status         object
alcohol_use            object
bmi                   float64
blood_group            object
income_level           object
insurance_status       object
treatment_type         object
hospital_type          object
days_hospitalized       int64
treatment_cost_inr    float64
outcome                object
death_flag              int64
cause_of_death         object
recovery_days         float64
follow_up_required     object
dtype: object

In [37]:
# Post-diagnosis columns (can't know at triage time)
# Plus irrelevant ID/location columns
DROP_COLS = [
    "patient_id",          # just an ID
    "age_group",           # redundant with age
    "state", "city",       # too granular
    "disease_name",        # post-diagnosis
    "diagnosis_date",      # post-diagnosis
    "year", "month",       # not useful for triage
    "treatment_type",      # post-diagnosis
    "hospital_type",       # post-diagnosis
    "days_hospitalized",   # post-diagnosis
    "treatment_cost_inr",  # post-diagnosis
    "outcome",             # post-diagnosis
    "death_flag",          # post-diagnosis
    "cause_of_death",      # post-diagnosis
    "recovery_days",       # post-diagnosis
    "follow_up_required",  # post-diagnosis
    "insurance_status",    # not clinical
    "income_level",        # not clinical
    "blood_group",         # not relevant for triage
]

df = df.drop(columns=DROP_COLS)
print("After drop:", df.shape)
print(df.columns.tolist())

After drop: (20000, 12)
['age', 'gender', 'region', 'urban_rural', 'disease_category', 'severity', 'season', 'symptoms', 'comorbidity', 'smoking_status', 'alcohol_use', 'bmi']


In [38]:
# Your plan exactly
df["alcohol_use"]   = df["alcohol_use"].fillna("Never")
df["comorbidity"]   = df["comorbidity"].fillna("None")

# Verify no more nulls
print(df.isnull().sum())

age                 0
gender              0
region              0
urban_rural         0
disease_category    0
severity            0
season              0
symptoms            0
comorbidity         0
smoking_status      0
alcohol_use         0
bmi                 0
dtype: int64


In [39]:
# First check what values exist
print(df["severity"].unique())

# Map to your 4 classes
severity_map = {
    "Mild":     "Low",
    "Moderate": "Medium",
    "Severe":   "High",
    "Critical": "Critical"
}

df["risk_level"] = df["severity"].map(severity_map)

# Verify no unmapped values
print(df["risk_level"].isna().sum())   # should be 0
print(df["risk_level"].value_counts())

# Drop original severity column
df = df.drop(columns=["severity"])

['Severe' 'Mild' 'Critical' 'Moderate']
0
risk_level
Medium      6516
High        6305
Low         4850
Critical    2329
Name: count, dtype: int64


In [40]:
# ============================================================
# SNIPPET 5 — Parse Symptoms into Binary Columns
# ============================================================

# Step 1 — Check raw data
print("Raw symptoms sample:")
print(df["symptoms"].head(10))
print("\nNull count:", df["symptoms"].isna().sum())

# Step 2 — Standardize symptom names to match agent's ALL_SYMPTOMS
SYMPTOM_MAP = {
    "severe headache":      "headache",
    "shortness of breath":  "breathlessness",
    "loss of smell/taste":  "loss_of_smell_or_taste",
    "loss of smell taste":  "loss_of_smell_or_taste",
    "unexplained weight loss": "weight_loss",
    "pain episodes":        "pain_episodes",
    "frequent infections":  "frequent_infections",
    "frequent urination":   "burning_urination",
    "high fever":           "fever",
    # add more here if you spot mismatches
}

def clean_symptom(s):
    s = s.strip().lower()
    s = s.replace("/", "_or_")   # fix forward slash
    s = s.replace(" ", "_")      # spaces to underscore
    s = s.replace("-", "_")      # hyphens to underscore
    return SYMPTOM_MAP.get(s.replace("_", " "), s)  # check map

# Step 3 — Apply cleaning and find all unique symptoms
all_symptoms = set()
df["symptoms_cleaned"] = df["symptoms"].apply(
    lambda x: [clean_symptom(s) for s in x.split(",")] if pd.notna(x) else []
)
df["symptoms_cleaned"].apply(lambda x: all_symptoms.update(x))

print("\nUnique symptoms after cleaning:")
print(sorted(all_symptoms))
print("Total unique symptoms:", len(all_symptoms))

# Step 4 — Create binary columns
for symptom in sorted(all_symptoms):
    df[f"symptom_{symptom}"] = df["symptoms_cleaned"].apply(
        lambda x: 1 if symptom in x else 0
    )

# Step 5 — Add derived count column
df["num_symptoms"] = df["symptoms_cleaned"].apply(len)

# Step 6 — Verify no invalid characters in column names
bad_cols = [c for c in df.columns if "/" in c or " " in c]
print("\nBad column names (should be empty):", bad_cols)

# Step 7 — Drop original and cleaned columns
df = df.drop(columns=["symptoms", "symptoms_cleaned"])

print("\nAfter symptom encoding:", df.shape)
print("Symptom columns created:", [c for c in df.columns if c.startswith("symptom_")])

Raw symptoms sample:
0                      Vomiting, Abdominal pain, Fever
1    Vomiting, Rash, Joint pain, Nausea, Severe Hea...
2    Dizziness, Chest pain, Swelling, Shortness of ...
3            Fever, Body ache, Nausea, Fatigue, Chills
4    Blurred vision, Dizziness, Unexplained weight ...
5      Wheezing, Fever, Cough, Sore throat, Chest pain
6    Wheezing, Fatigue, Chest pain, Loss of smell/t...
7         Swelling, Pain episodes, Frequent infections
8                      Fatigue, Wheezing, Fever, Cough
9    Swelling, Blurred vision, Numbness, Frequent u...
Name: symptoms, dtype: object

Null count: 0

Unique symptoms after cleaning:
['abdominal_pain', 'anemia', 'appetite_changes', 'blurred_vision', 'body_ache', 'bone_deformities', 'breathlessness', 'burning_urination', 'chest_pain', 'chills', 'cough', 'cramps', 'dehydration', 'delayed_growth', 'diarrhea', 'dizziness', 'excessive_worry', 'fatigue', 'fever', 'frequent_infections', 'headache', 'irritability', 'joint_pain', 'loss_o

In [41]:
# Should all be 0
print("Nulls in symptom cols:", df[[c for c in df.columns if c.startswith("symptom_")]].isnull().sum().sum())

# Should be empty list
print("Bad col names:", [c for c in df.columns if "/" in c or " " in c])

Nulls in symptom cols: 0
Bad col names: []


In [42]:
# Same approach as symptoms
print(df["comorbidity"].head(10))

all_conditions = set()
df["comorbidity"].dropna().str.split(",").apply(
    lambda x: all_conditions.update([c.strip().lower().replace(" ", "_") for c in x])
)

# Remove "none" from conditions set
all_conditions.discard("none")
print("Unique conditions found:", sorted(all_conditions))

# Create binary columns
for condition in sorted(all_conditions):
    df[f"condition_{condition}"] = df["comorbidity"].apply(
        lambda x: 1 if pd.notna(x) and condition in x.lower().replace(" ", "_") else 0
    )

# Add derived columns
df["has_pre_existing"] = df["comorbidity"].apply(
    lambda x: 0 if pd.isna(x) or x.strip().lower() == "none" else 1
)
df["num_conditions"] = df[[c for c in df.columns if c.startswith("condition_")]].sum(axis=1)
df["num_symptoms"]   = df[[c for c in df.columns if c.startswith("symptom_")]].sum(axis=1)

# Drop original comorbidity column
df = df.drop(columns=["comorbidity"])
print("After comorbidity encoding:", df.shape)

0              None
1    Kidney Disease
2     Heart Disease
3              None
4    Kidney Disease
5          Multiple
6    Kidney Disease
7              None
8              None
9          Multiple
Name: comorbidity, dtype: object
Unique conditions found: ['diabetes', 'heart_disease', 'hypertension', 'kidney_disease', 'multiple', 'obesity']
After comorbidity encoding: (20000, 61)


In [43]:
# Check remaining object columns
cat_cols = df.select_dtypes(include="object").columns.tolist()
cat_cols.remove("risk_level")  # don't encode target yet
print("Categorical columns to encode:", cat_cols)

# These should just be:
# gender, region, urban_rural, disease_category,
# season, smoking_status, alcohol_use

# Label encode each
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {le.classes_}")

Categorical columns to encode: ['gender', 'region', 'urban_rural', 'disease_category', 'season', 'smoking_status', 'alcohol_use']
gender: ['Female' 'Male' 'Other']
region: ['Central' 'East' 'North' 'Northeast' 'South' 'West']
urban_rural: ['Rural' 'Semi-Urban' 'Urban']
disease_category: ['Genetic' 'Infectious' 'Mental Health' 'Non-Communicable' 'Respiratory'
 'Vector-Borne' 'Waterborne']
season: ['Monsoon' 'Post-Monsoon' 'Summer' 'Winter']
smoking_status: ['Current' 'Former' 'Never']
alcohol_use: ['Heavy' 'Never' 'Occasional' 'Regular']


In [44]:
target_encoder = LabelEncoder()
df["risk_level_encoded"] = target_encoder.fit_transform(df["risk_level"])

print("Classes:", target_encoder.classes_)
# → ['Critical', 'High', 'Low', 'Medium']
# → [0,          1,      2,     3]

# Save label encoder now
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(target_encoder, f)

print("✅ label_encoder.pkl saved")

Classes: ['Critical' 'High' 'Low' 'Medium']
✅ label_encoder.pkl saved


In [45]:
# Separate features and target
X = df.drop(columns=["risk_level", "risk_level_encoded"])
y = df["risk_level_encoded"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:\n", df["risk_level"].value_counts())
print("Any nulls left:", X.isnull().sum().sum())  # should be 0
print("Feature columns:\n", X.columns.tolist())

Feature shape: (20000, 60)
Target shape: (20000,)
Target distribution:
 risk_level
Medium      6516
High        6305
Low         4850
Critical    2329
Name: count, dtype: int64
Any nulls left: 0
Feature columns:
 ['age', 'gender', 'region', 'urban_rural', 'disease_category', 'season', 'smoking_status', 'alcohol_use', 'bmi', 'symptom_abdominal_pain', 'symptom_anemia', 'symptom_appetite_changes', 'symptom_blurred_vision', 'symptom_body_ache', 'symptom_bone_deformities', 'symptom_breathlessness', 'symptom_burning_urination', 'symptom_chest_pain', 'symptom_chills', 'symptom_cough', 'symptom_cramps', 'symptom_dehydration', 'symptom_delayed_growth', 'symptom_diarrhea', 'symptom_dizziness', 'symptom_excessive_worry', 'symptom_fatigue', 'symptom_fever', 'symptom_frequent_infections', 'symptom_headache', 'symptom_irritability', 'symptom_joint_pain', 'symptom_loss_of_appetite', 'symptom_loss_of_interest', 'symptom_loss_of_smell_or_taste', 'symptom_muscle_pain', 'symptom_nausea', 'symptom_numbnes

In [46]:
df = df.drop(columns=["condition_multiple"])

In [52]:
# See all 60 columns and their dtypes
print(df.dtypes.to_string())

age                                 int64
gender                              int64
region                              int64
urban_rural                         int64
disease_category                    int64
season                              int64
smoking_status                      int64
alcohol_use                         int64
bmi                               float64
risk_level                         object
symptom_abdominal_pain              int64
symptom_anemia                      int64
symptom_appetite_changes            int64
symptom_blurred_vision              int64
symptom_body_ache                   int64
symptom_bone_deformities            int64
symptom_breathlessness              int64
symptom_burning_urination           int64
symptom_chest_pain                  int64
symptom_chills                      int64
symptom_cough                       int64
symptom_cramps                      int64
symptom_dehydration                 int64
symptom_delayed_growth            

In [53]:
for col in ["gender", "region", "urban_rural", "smoking_status", 
            "alcohol_use", "disease_category", "season"]:
    print(f"{col}: {label_encoders[col].classes_}")

gender: ['Female' 'Male' 'Other']
region: ['Central' 'East' 'North' 'Northeast' 'South' 'West']
urban_rural: ['Rural' 'Semi-Urban' 'Urban']
smoking_status: ['Current' 'Former' 'Never']
alcohol_use: ['Heavy' 'Never' 'Occasional' 'Regular']
disease_category: ['Genetic' 'Infectious' 'Mental Health' 'Non-Communicable' 'Respiratory'
 'Vector-Borne' 'Waterborne']
season: ['Monsoon' 'Post-Monsoon' 'Summer' 'Winter']


In [58]:
# ============================================================
# SNIPPET 10 — Train XGBoost + Save Artifacts
# ============================================================

import pickle
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

# ============================================================
# STEP 1 — Save Processed Data as CSV
# ============================================================

df.to_csv("processed_data.csv", index=False)
print("✅ processed_data.csv saved")
print(f"   Shape: {df.shape}")

# ============================================================
# STEP 2 — Separate Features and Target
# ============================================================

X = df.drop(columns=["risk_level", "risk_level_encoded"])
y = df["risk_level_encoded"]

FEATURE_COLUMNS = X.columns.tolist()
print(f"\n✅ Features: {len(FEATURE_COLUMNS)} columns")
print(f"   Target distribution:\n{df['risk_level'].value_counts()}")

# ============================================================
# STEP 3 — Train / Test Split (stratified for imbalance)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # ensures Critical class is represented in both splits
)

print(f"\n✅ Split complete")
print(f"   Train: {X_train.shape[0]} rows")
print(f"   Test : {X_test.shape[0]} rows")

# ============================================================
# STEP 4 — Handle Class Imbalance
# ============================================================

# Count samples per class
class_counts = y_train.value_counts().sort_index()
print(f"\nClass counts in train:\n{class_counts}")

# Compute weight for each sample
# Gives more weight to underrepresented classes (Critical)
total = len(y_train)
sample_weights = y_train.map(
    lambda c: total / (len(class_counts) * class_counts[c])
)

# ============================================================
# STEP 5 — Train XGBoost
# ============================================================

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("\n🔄 Training XGBoost...")

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=50        # print every 50 rounds
)

print("✅ Training complete")

# ============================================================
# STEP 6 — Evaluate
# ============================================================

y_pred = model.predict(X_test)

# Get label encoder to decode class names
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

class_names = le.classes_

print("\n" + "=" * 60)
print("📊 CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=class_names))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

f1 = f1_score(y_test, y_pred, average="weighted")
print(f"\n✅ Weighted F1 Score: {f1:.4f}")

# ============================================================
# STEP 7 — Save model.pkl
# ============================================================

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("\n✅ model.pkl saved")

# ============================================================
# STEP 8 — Save feature columns (important for prediction)
# ============================================================

with open("feature_columns.pkl", "wb") as f:
    pickle.dump(FEATURE_COLUMNS, f)

print("✅ feature_columns.pkl saved")

# ============================================================
# STEP 9 — Verify saved artifacts
# ============================================================

print("\n" + "=" * 60)
print("🔍 VERIFICATION — Loading and testing saved artifacts")
print("=" * 60)

with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    loaded_le = pickle.load(f)

with open("feature_columns.pkl", "rb") as f:
    loaded_features = pickle.load(f)

# Quick prediction test
sample = X_test.iloc[[0]]
risk_code = loaded_model.predict(sample)[0]
risk_label = loaded_le.inverse_transform([risk_code])[0]
probabilities = loaded_model.predict_proba(sample)[0]
confidence = {
    cls: round(float(prob) * 100, 1)
    for cls, prob in zip(loaded_le.classes_, probabilities)
}

print(f"Sample prediction  : {risk_label}")
print(f"Confidence         : {confidence}")
print(f"Feature cols match : {loaded_features == FEATURE_COLUMNS}")

print("\n" + "=" * 60)
print("✅ ALL ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 60)
print("   model.pkl")
print("   label_encoder.pkl")
print("   feature_columns.pkl")
print("   processed_data.csv")
print("=" * 60)

✅ processed_data.csv saved
   Shape: (20000, 61)

✅ Features: 59 columns
   Target distribution:
risk_level
Medium      6516
High        6305
Low         4850
Critical    2329
Name: count, dtype: int64

✅ Split complete
   Train: 16000 rows
   Test : 4000 rows

Class counts in train:
risk_level_encoded
0    1863
1    5044
2    3880
3    5213
Name: count, dtype: int64

🔄 Training XGBoost...
[0]	validation_0-mlogloss:1.38451
[50]	validation_0-mlogloss:1.35294
[100]	validation_0-mlogloss:1.35057
[150]	validation_0-mlogloss:1.34984
[200]	validation_0-mlogloss:1.35032
[250]	validation_0-mlogloss:1.35170
[299]	validation_0-mlogloss:1.35261
✅ Training complete

📊 CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Critical       0.16      0.27      0.20       466
        High       0.39      0.31      0.35      1261
         Low       0.33      0.40      0.36       970
      Medium       0.37      0.31      0.34      1303

    accuracy                           0.